In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
# add the project root to the path so the src package can be imported
sys.path.append(os.path.abspath('..'))

import yaml
from ultralytics import YOLO
from src.model_utils import smart_predict,compare_best_vs_last
# on_fit_epoch_end will prompt for the current stage number when it's called
from src.model_callbacks import on_fit_epoch_end
from src.generate_report import generate_report

%matplotlib inline

Patience Limit Is 20 Loaded Successfuly from the config file!


In [ ]:
with open('../configs/stage2.yaml','r') as f:

    stage2_config = yaml.safe_load(f)


with open(stage2_config['model_args']['data'],'r') as f:

    enum_data_yaml = yaml.safe_load(f)


with open('../configs/trained_models.yaml','r') as f:

    trained_models_config = yaml.safe_load(f)

In [3]:
stage2_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration/train_quadrant_enumeration.json',
  's2_main_path': '..\\Data\\Processed\\Stage 2 (Enumeration Detection)',
  'runs_s2_output': '..\\Runs\\Stage 2'},
 'model_args': {'data': '..\\Data\\Processed\\Stage 2 (Enumeration Detection)\\data.yaml',
  'model': '../Models/yolo26s.pt',
  'epochs': 70,
  'save': True,
  'imgsz': 1280,
  'batch': 16,
  'patience': 20,
  'optimizer': 'AdamW',
  'lr0': 0.001,
  'lrf': 0.01,
  'plots': True,
  'verbose': True,
  'device': 'cuda',
  'workers': 4,
  'project': 'Runs',
  'name': 'Stage 2',
  'save_dir': '..\\Runs\\Stage 2',
  'auto_augment': 'None',
  'augment': True,
  'mosaic': 0.0,
  'mixup': 0.0,
  'copy_paste': 0.0,
  'cutmix': 0.0,
  'hsv_h': 0.015,
  'hsv_s': 0.4,
  'hsv_v': 0.4,
  'degrees': 5.0,
  'translate': 0.05,
  'scale': 0.1,
  'shear':

In [4]:
enum_data_yaml

{'train': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\train\\images',
 'val': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\valid\\images',
 'test': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\test\\images',
 'nc': 8,
 'names': [0, 1, 2, 3, 4, 5, 6, 7]}

In [5]:
trained_models_config

{'quadrant_detection_model': {'best': '..\\Runs\\Stage 1\\weights\\best.pt',
  'last': '..\\Runs\\Stage 1\\weights\\last.pt'},
 'enumeration_detection_model': {'best': '..\\Runs\\Stage 2\\weights\\best.pt',
  'last': '..\\Runs\\Stage 2\\weights\\last.pt'},
 'disease_classification_model': {'best': '..\\Runs\\Stage 3\\weights\\best.pt',
  'last': '..\\Runs\\Stage 3\\weights\\last.pt'}}

In [ ]:
# load the base checkpoint specified in the config, this is the starting point before training
yolo_model = YOLO(stage2_config['model_args']['model'])

In [ ]:
yolo_model.add_callback('on_fit_epoch_end',on_fit_epoch_end)

# # training already ran once; left here commented out for reference
# yolo_model.train(**stage2_config['model_args'])

In [ ]:
# compare the best vs last checkpoint on the enumeration task using the test split
results = compare_best_vs_last(trained_models_config,
                                'enum',
                                stage2_config['paths']['original_images_path'],
                                os.path.join(stage2_config['paths']['s2_main_path'], 'test_df (splitted).pkl'),
                                os.path.join(stage2_config['paths']['runs_s2_output'], 'results.csv'))

In [ ]:
# load the checkpoint saved at the best of teeth training, this is the model we'll actually use for inference
teeth_model_detection = YOLO(os.path.join(stage2_config['paths']['runs_s2_output'], 'weights', 'last.pt'))

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'])

In [ ]:
smart_predict(teeth_model_detection,os.path.join(stage2_config['paths']['diagnosis_quadrants_train_path'], 'train', 'images'))

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'],save_output=True,save_dir=os.path.join(stage2_config['paths']['runs_s2_output'],'Test Outputs Predictions'))

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'],save_crop_output_image=True,save_dir=os.path.join(stage2_config['paths']['runs_s2_output'],'Test Outputs Predictions'))

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'],show_true_boxes=True)

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'],show_true_boxes=True,apply_custom_draw_box=True)

In [ ]:
# overlay the ground-truth boxes next to the predictions for a visual comparison
smart_predict(teeth_model_detection,enum_data_yaml['test'],show_true_boxes=True)

In [ ]:
# left disabled for now; uncomment to export a training report PDF
generate_report(stage2_config['paths']['runs_s2_output'], 
                os.path.join(stage2_config['paths']['runs_s2_output'], 'YOLO Training Report (Stage 2).pdf'),
                title='YOLO Training Report (Stage 2)')